## 4 - Modelo TWFE

Estima el modelo TWFE en primeras diferencias sobre el `PanelModelo`, con la especificación (formas, rezagos y combinaciones) del archivo de decisiones. Produce los efectos por combinación con IC 95, el aporte por capacidad, la atribución YoY, el export de β y varianzas, y las pruebas de validación y robustez.

In [ ]:
pip install pyfixest

# Parámetros

Único bloque a editar por embotellador-país.

In [ ]:
BU = 'MEX'

In [ ]:
# Inferencia: CRV1 = cluster por id_cliente (reporte final, lento). HC1 = robusto (iteración rápida)
VCOV_TIPO = 'HC1'

# Meses del período de referencia para el aporte por capacidad
MESES_REFERENCIA = 3

# Rezagos con patrón de reversión a la media: se reportan pero se excluyen de los agregados
REZAGOS_FUERA_BUNDLE = []

# Robustez
RUN_R_CLUSTER = False          # sensibilidad de errores estándar con CRV1 (costoso)
PLACEBO_DESPLAZAMIENTOS = (-3, -6)
SEED = 42

# Setup y Lectura de Datos

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import gc
import pyfixest as pf
import matplotlib.pyplot as plt
from pyspark.sql import functions as sf
from itertools import permutations

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")

PATH_MODELO     = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/PanelModelo/parquet/"
PATH_DECISIONES = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/Modelo/capacidades"

In [ ]:
# Decisiones del modelo: capacidades, formas, rezagos y combinaciones validadas
rows = spark.read.text(PATH_DECISIONES).collect()
DECISIONES = json.loads('\n'.join(r[0] for r in rows))

FORMA       = DECISIONES['formas']                    
REZAGOS     = DECISIONES['rezagos']                   
LISTA_C9    = DECISIONES['combinaciones_estructura'] 
CAPACIDADES = list(FORMA.keys())

print(f"BU: {DECISIONES.get('bu')} | universo: {DECISIONES.get('universo')} | escala: {DECISIONES.get('convencion_proporciones')}")
print(f"Capacidades ({len(CAPACIDADES)}): {CAPACIDADES}")
print(f"Formas:      {FORMA}")
print(f"Rezagos t-1: {REZAGOS}")
print(f"Combinaciones ({len(LISTA_C9)}):")
for i, combo in enumerate(LISTA_C9, 1):
    print(f"  {i:>2}. {' x '.join(combo)}")

In [ ]:
# Carga del PanelModelo y variables derivadas
cols_carga = ['id_cliente', 'period_id', 'ingreso_neto_core_real', 'tamano_cliente', 'subcanal'] + CAPACIDADES
t0 = time.time()
panel_pd = spark.read.parquet(PATH_MODELO).select(*cols_carga).toPandas()
spark.catalog.clearCache()

n_pre = len(panel_pd)
panel_pd = panel_pd.dropna(subset=['ingreso_neto_core_real'] + CAPACIDADES)
panel_pd['id_cliente'] = panel_pd['id_cliente'].astype('category')
panel_pd['period_id']  = panel_pd['period_id'].astype('int32')
for cap in CAPACIDADES:
    panel_pd[cap] = panel_pd[cap].astype('float32')

panel_pd['log_y'] = np.log(panel_pd['ingreso_neto_core_real'].clip(lower=0.01)).astype('float32')
panel_pd = panel_pd.drop(columns=['ingreso_neto_core_real'])
gc.collect()

print(f"PDVs:    {panel_pd['id_cliente'].nunique():,}")
print(f"Filas:   {len(panel_pd):,} (dropna -{n_pre - len(panel_pd):,})")
print(f"Periodo: {panel_pd['period_id'].min()} a {panel_pd['period_id'].max()}")
print(f"Carga:   {time.time()-t0:.1f}s | memoria panel: {panel_pd.memory_usage(deep=True).sum()/1e9:.2f} GB")
for cap in CAPACIDADES:
    print(f"  {cap:<18} [{panel_pd[cap].min():.2f}, {panel_pd[cap].max():.2f}]")

# Especificación del modelo

In [ ]:
# Helper de estimación para centralizar el tipo de errores estándar
def estimar(formula, data, vcov_tipo=None):
    if vcov_tipo is None:
        vcov_tipo = VCOV_TIPO
    kwargs = {"fml": formula, "data": data}
    if vcov_tipo == "CRV1":
        kwargs["vcov"] = {"CRV1": "id_cliente"}
    elif vcov_tipo == "HC1":
        kwargs["vcov"] = "HC1"
    return pf.feols(**kwargs)

In [ ]:
# Términos del modelo según la forma de cada capacidad; las cuadráticas se desdoblan en x y x²
EPS = 1
CUADRATICAS = [c for c in CAPACIDADES if FORMA[c] == 'cuadratica']

def termino_forma(cap):
    return f'log_{cap}' if FORMA[cap] == 'log' else cap

for cap in CAPACIDADES:
    if FORMA[cap] == 'log' and f'log_{cap}' not in panel_pd.columns:
        panel_pd[f'log_{cap}'] = np.log(panel_pd[cap] + EPS).astype('float32')
    if FORMA[cap] == 'cuadratica' and f'{cap}_sq' not in panel_pd.columns:
        panel_pd[f'{cap}_sq'] = (panel_pd[cap].astype('float64') ** 2).astype('float32')

def construir_terminos(combo):
    quads = [c for c in combo if c in CUADRATICAS]
    if not quads:
        return [':'.join(termino_forma(c) for c in combo)]
    q = quads[0]
    otras = [c for c in combo if c != q]
    base = ':'.join(termino_forma(c) for c in otras)
    return [f'{base}:{q}' if base else q, f'{base}:{q}_sq' if base else f'{q}_sq']

TERMINOS = []
for combo in LISTA_C9:
    for t in construir_terminos(combo):
        if t not in TERMINOS:
            TERMINOS.append(t)

# Rezagos t-1: solo capacidades con término contemporáneo en el modelo (regla del rezago huérfano)
LAGS = []
huerfanos = [c for c in REZAGOS if not any(c in combo for combo in LISTA_C9)]
for cap in [c for c in REZAGOS if c not in huerfanos]:
    col = termino_forma(cap) if FORMA[cap] != 'cuadratica' else cap
    base = panel_pd.set_index(['id_cliente', 'period_id'])[col]
    idx  = pd.MultiIndex.from_arrays([panel_pd['id_cliente'], panel_pd['period_id'] - 1])
    panel_pd[f'{cap}_l1'] = pd.Series(base.reindex(idx).values).fillna(0).astype('float32').values
    LAGS.append(f'{cap}_l1')
del base, idx
gc.collect()

if huerfanos:
    print(f"Rezagos huérfanos removidos (capacidad sin término contemporáneo): {huerfanos}")
print(f"Términos de combinaciones: {len(TERMINOS)} | rezagos: {len(LAGS)}")
print(f"Cuadráticas desdobladas: {CUADRATICAS}")

# Modelo Primeras Diferencias (FD)

In [ ]:
#Helper de estimacion para centralizar vcov

def estimar(formula, data, vcov_tipo=None):
    if vcov_tipo is None:
        vcov_tipo = VCOV_TIPO
    kwargs = {"fml": formula, "data": data}
    if vcov_tipo == "CRV1":
        kwargs["vcov"] = {"CRV1": "id_cliente"}
    elif vcov_tipo == "HC1":
        kwargs["vcov"] = "HC1"
    # IID o cualquier otro: sin vcov, pyfixest usa iid por defecto
    return pf.feols(**kwargs)

In [ ]:
# Interacciones y primeras diferencias de y y de cada término, todo en float32
def _dname(t):
    return 'd_' + t.replace(':', '__')

for t in TERMINOS + LAGS:
    if ':' in t and t not in panel_pd.columns:
        v = np.ones(len(panel_pd), dtype='float64')
        for q in t.split(':'):
            v *= panel_pd[q].values
        panel_pd[t] = v.astype('float32')
        del v

for col in ['log_y'] + TERMINOS + LAGS:
    base = panel_pd.set_index(['id_cliente', 'period_id'])[col]
    idx  = pd.MultiIndex.from_arrays([panel_pd['id_cliente'], panel_pd['period_id'] - 1])
    nombre = 'd_y' if col == 'log_y' else _dname(col)
    panel_pd[nombre] = (panel_pd[col].values - base.reindex(idx).values).astype('float32')
del base, idx
gc.collect()

D_TERMS = [_dname(t) for t in TERMINOS + LAGS]
print(f"Memoria panel: {panel_pd.memory_usage(deep=True).sum()/1e9:.2f} GB")

In [ ]:
# Estimación FD con salvaguarda de identificación (SE>10); se extraen β, V y tabla y se libera el modelo
UMBRAL_SE = 10.0
COLS_DP = ['id_cliente', 'period_id', 'tamano_cliente', 'subcanal', 'd_y'] + D_TERMS
dp = panel_pd[COLS_DP].dropna(subset=['d_y'] + D_TERMS)
gc.collect()

t0 = time.time()
modelo_fd = estimar(f"d_y ~ {' + '.join(D_TERMS)} | id_cliente + period_id", dp)
td_fd = modelo_fd.tidy()
NO_IDENTIFICADOS = sorted(t for t in td_fd.index if abs(float(td_fd.loc[t, 'Std. Error'])) > UMBRAL_SE)
if NO_IDENTIFICADOS:
    print(f"Términos no identificados (SE>{UMBRAL_SE}), se excluyen y re-estima: {NO_IDENTIFICADOS}")
    D_TERMS = [d for d in D_TERMS if d not in NO_IDENTIFICADOS]
    LAGS = [lt for lt in LAGS if _dname(lt) in D_TERMS]
    del modelo_fd
    gc.collect()
    modelo_fd = estimar(f"d_y ~ {' + '.join(D_TERMS)} | id_cliente + period_id", dp)
    td_fd = modelo_fd.tidy()

co_fd  = modelo_fd.coef()
V_fd   = np.asarray(modelo_fd._vcov)
NOMBRES = list(co_fd.index)
print(f"Modelo FD estimado en {time.time()-t0:.1f}s sobre {len(dp):,} diferencias PDV-mes")
modelo_fd.summary()
del modelo_fd
gc.collect()

# Efectos por combinación e IC 95

In [ ]:
# Efectos en % de venta: exp(w·β)-1 con IC por método delta (w'Vw); bundle contemporáneo y de largo plazo
REF = {}
for cap in CAPACIDADES:
    raw = panel_pd[cap]
    act = raw > 0
    if not act.any():
        REF[cap] = 0.0
    elif FORMA[cap] == 'log':
        REF[cap] = float(np.log(raw[act].median() + EPS))
    elif FORMA[cap] == 'binaria':
        REF[cap] = 1.0
    else:
        REF[cap] = float(raw[act].median())
del raw, act

def pares_combo(combo):
    quads = [c for c in combo if c in CUADRATICAS]
    if quads:
        q = quads[0]
        otras = [c for c in combo if c != q]
        pio = float(np.prod([REF[c] for c in otras])) if otras else 1.0
        base = ':'.join(termino_forma(c) for c in otras)
        t1 = f'{base}:{q}' if base else q
        t2 = f'{base}:{q}_sq' if base else f'{q}_sq'
        return [(t1, pio * REF[q]), (t2, pio * REF[q] ** 2)]
    t = construir_terminos(combo)[0]
    return [(t, float(np.prod([REF[c] for c in combo])))]

def w_efecto(pares):
    w = np.zeros(len(NOMBRES)); ok = False
    for t, pi in pares:
        d = _dname(t)
        if d in NOMBRES:
            w[NOMBRES.index(d)] += pi; ok = True
    if not ok:
        return None
    b = float(w @ co_fd.values); se = float(np.sqrt(max(w @ V_fd @ w, 0)))
    return ((np.exp(b)-1)*100, (np.exp(b-1.96*se)-1)*100, (np.exp(b+1.96*se)-1)*100, b)

print(f"{'combinación':<52}{'efecto %':>10}{'IC 95%':>22}")
BUNDLE_W = []
for combo in LISTA_C9:
    pares = pares_combo(combo)
    r = w_efecto(pares)
    if r is None:
        print(f"{' x '.join(combo):<52}{'no estimable':>10}")
        continue
    print(f"{' x '.join(combo):<52}{r[0]:>+9.2f}%   [{r[1]:+.2f}; {r[2]:+.2f}]")
    BUNDLE_W += pares
for lt in LAGS:
    cap = lt[:-3]
    r = w_efecto([(lt, REF[cap])])
    if r is None:
        continue
    marca_rev = '  (negativo: evaluar patrón de reversión)' if r[3] < 0 else ''
    fuera = '  (fuera de agregados)' if cap in REZAGOS_FUERA_BUNDLE else ''
    print(f"{cap + ' (t-1)':<52}{r[0]:>+9.2f}%   [{r[1]:+.2f}; {r[2]:+.2f}]{marca_rev}{fuera}")
    if cap not in REZAGOS_FUERA_BUNDLE:
        BUNDLE_W.append((lt, REF[cap]))

bc = w_efecto([(t, p) for t, p in BUNDLE_W if not t.endswith('_l1')])
bl = w_efecto(BUNDLE_W)
print(f"\nBundle contemporáneo: {bc[0]:+.1f}%  [{bc[1]:+.1f}; {bc[2]:+.1f}]")
print(f"Bundle largo plazo:   {bl[0]:+.1f}%  [{bl[1]:+.1f}; {bl[2]:+.1f}]")

# Aporte por capacidad en el período de referencia

In [ ]:
# Aporte del nivel del período de referencia, sobre activos y sobre el total; los aportes se solapan entre capacidades, no sumarlos
pmax_ref = int(panel_pd['period_id'].max())
ref_mask = (panel_pd['period_id'] > pmax_ref - MESES_REFERENCIA).values

def cap_de_factor(f):
    f = f.replace('_l1', '').replace('_sq', '')
    return f[4:] if f.startswith('log_') else f

print(f"Período de referencia: últimos {MESES_REFERENCIA} meses")
print(f"{'capacidad':<18}{'s/activos':>12}{'s/total':>12}{'PDVs activos':>14}")
for cap in CAPACIDADES:
    terms_cap = [t for t in TERMINOS + LAGS if cap in [cap_de_factor(f) for f in t.split(':')]]
    if not terms_cap:
        continue
    act = (panel_pd.loc[ref_mask, cap] > 0).values
    n_act = panel_pd.loc[ref_mask, 'id_cliente'][act].nunique() if act.any() else 0
    lp_act = lp_tot = 0.0
    for t in terms_cap:
        d = _dname(t)
        if d not in NOMBRES:
            continue
        v = panel_pd.loc[ref_mask, t].values.astype('float64')
        lp_tot += float(co_fd[d]) * float(v.mean())
        lp_act += float(co_fd[d]) * (float(v[act].mean()) if act.any() else 0.0)
        del v
    print(f"{cap:<18}{(np.exp(lp_act)-1)*100:>+11.2f}%{(np.exp(lp_tot)-1)*100:>+11.2f}%{n_act:>14,}")
gc.collect()

# Atribución YoY por movimiento

In [ ]:
# Aporte del año por capacidad: β·Δx̄ acreditado al factor que se movió (descomposición simétrica del Δ de productos)
tT = int(panel_pd['period_id'].max())
mB = (panel_pd['period_id'] > tT - 12).values
mA = ((panel_pd['period_id'] <= tT - 12) & (panel_pd['period_id'] > tT - 24)).values
COLS_MED = list(dict.fromkeys(TERMINOS + LAGS + [f for t in TERMINOS + LAGS for f in t.split(':')]))
medB = panel_pd.loc[mB, COLS_MED].mean()
medA = panel_pd.loc[mA, COLS_MED].mean()
gc.collect()

aporte_cap = {}
for t in TERMINOS + LAGS:
    d = _dname(t)
    if d not in NOMBRES:
        continue
    d_lp = float(co_fd[d]) * float(medB[t] - medA[t])
    fs = t.split(':')
    if len(fs) == 1:
        c = cap_de_factor(fs[0])
        aporte_cap[c] = aporte_cap.get(c, 0.0) + d_lp
        continue
    a = [float(medA[f]) for f in fs]
    b = [float(medB[f]) for f in fs]
    parts = [0.0] * len(fs)
    P = list(permutations(range(len(fs))))
    for pm in P:
        cur = list(a)
        for i in pm:
            antes = float(np.prod(cur)); cur[i] = b[i]
            parts[i] += (float(np.prod(cur)) - antes) / len(P)
    dP = sum(parts)
    for i, f in enumerate(fs):
        sh = parts[i] / dP if abs(dP) > 1e-12 else 1.0 / len(fs)
        c = cap_de_factor(f)
        aporte_cap[c] = aporte_cap.get(c, 0.0) + d_lp * sh

print('Aporte YoY por capacidad (reparto por movimiento):')
for c, v in sorted(aporte_cap.items(), key=lambda kv: -abs(kv[1])):
    print(f'  {c:16s} {(np.exp(v)-1)*100:+.2f}%')
print(f'  {"TOTAL":16s} {(np.exp(sum(aporte_cap.values()))-1)*100:+.2f}%')

# Descomposición YoY: mercado, capacidades y composición

Identidad contable del modelo auxiliar en niveles; se reporta sin intervalo. Los efectos oficiales son los del modelo en primeras diferencias.

In [ ]:
# Modelo auxiliar en niveles: materializa los efectos fijos de cliente (α) y de período (γ), calcula la descomposición y libera todo al final
FACTORES_NIV = list(dict.fromkeys([f for t in TERMINOS for f in t.split(':')]))
panel_niv = panel_pd[['id_cliente', 'period_id', 'log_y'] + FACTORES_NIV + LAGS]
t0 = time.time()
modelo_niv = estimar(f"log_y ~ {' + '.join(TERMINOS + LAGS)} | id_cliente + period_id", panel_niv)
print(f"Auxiliar en niveles estimado en {time.time()-t0:.1f}s")
co_niv = modelo_niv.coef()
fe = modelo_niv.fixef()
del modelo_niv, panel_niv
gc.collect()

kA = [k for k in fe if 'id_cliente' in k][0]
kT = [k for k in fe if 'period_id' in k][0]
alpha = pd.Series(fe[kA])
gamma = pd.Series(fe[kT])
serie_a = panel_pd['id_cliente'].astype(str).map(alpha).astype('float64')
serie_g = panel_pd['period_id'].astype(str).map(gamma).astype('float64')

d_obs = float(panel_pd.loc[mB, 'log_y'].mean()) - float(panel_pd.loc[mA, 'log_y'].mean())
d_g   = float(serie_g[mB].mean()) - float(serie_g[mA].mean())
d_a   = float(serie_a[mB].mean()) - float(serie_a[mA].mean())
d_x   = 0.0
for t in TERMINOS + LAGS:
    if t not in co_niv.index:
        continue
    if t.endswith('_l1') and t[:-3] in REZAGOS_FUERA_BUNDLE:
        continue
    d_x += float(co_niv[t]) * float(medB[t] - medA[t])
resid = d_obs - d_g - d_a - d_x

print(f"Δ venta observado YoY: {(np.exp(d_obs)-1)*100:+.2f}%")
print(f"  Mercado (γ):         {(np.exp(d_g)-1)*100:+.2f}%")
print(f"  Capacidades (βΔx̄):   {(np.exp(d_x)-1)*100:+.2f}%")
print(f"  Composición (α):     {(np.exp(d_a)-1)*100:+.2f}%")
print(f"  Residuo:             {(np.exp(resid)-1)*100:+.2f}%")

g_curva = gamma.copy()
g_curva.index = g_curva.index.astype(int)
g_curva = g_curva.sort_index()
plt.figure(figsize=(10, 3.5))
plt.plot(g_curva.index, (np.exp(g_curva - g_curva.iloc[0]) - 1) * 100, marker='o', ms=3)
plt.axhline(0, color='gray', lw=0.7)
plt.title('Curva de mercado: efecto fijo de período (γ) acumulado vs primer mes')
plt.ylabel('%'); plt.xlabel('period_id')
plt.tight_layout(); plt.show()

del fe, alpha, gamma, serie_a, serie_g, g_curva
gc.collect()

In [ ]:
# Libera las columnas de interacción en niveles: aguas abajo solo se usan las diferencias d_*
cols_drop = [t for t in TERMINOS if ':' in t and t in panel_pd.columns]
panel_pd = panel_pd.drop(columns=cols_drop)
gc.collect()
print(f"Columnas liberadas: {len(cols_drop)} | memoria panel: {panel_pd.memory_usage(deep=True).sum()/1e9:.2f} GB")

# Export del modelo

In [ ]:
# Export de β, matriz de varianzas y referencias del modelo oficial: cualquier combinación w de términos
# tiene efecto exp(w·β)-1 con IC95 = exp(w·β ± 1.96·sqrt(w'Vw))-1, sin re-estimar
export = {
    'bu': DECISIONES.get('bu'),
    'modelo': 'TWFE en primeras diferencias',
    'vcov_tipo': VCOV_TIPO,
    'terms': NOMBRES,
    'coef': [float(co_fd[t]) for t in NOMBRES],
    'V': V_fd.tolist(),
    'referencias': REF,
    'formas': FORMA,
    'rezagos_en_modelo': [lt[:-3] for lt in LAGS],
    'rezagos_fuera_bundle': REZAGOS_FUERA_BUNDLE,
    'no_identificados': NO_IDENTIFICADOS,
    'fecha_generacion': pd.Timestamp.now().isoformat(),
}
RUTA_EXPORT = f"abfss://{containerName}@{storageAccountName}.dfs.core.windows.net/CTG/{BU}/Modelo/twfe_fd"
(spark.createDataFrame([(json.dumps(export, indent=2,
                                    default=lambda o: o.item() if hasattr(o, 'item') else str(o)),)], ['contenido'])
      .coalesce(1)
      .write.mode('overwrite')
      .text(RUTA_EXPORT))
print(f"Export escrito: {RUTA_EXPORT} | {len(NOMBRES)} términos, matriz {V_fd.shape[0]}x{V_fd.shape[1]}")

# Validación y robustez

In [ ]:
# Helpers: re-estimación de la spec FD sobre un subconjunto y bundle de un modelo dado
def estimar_fd(data, terms=None, vcov_tipo=None):
    terms = terms if terms is not None else D_TERMS
    return estimar(f"d_y ~ {' + '.join(terms)} | id_cliente + period_id", data, vcov_tipo)

def bundle_de(modelo):
    co = modelo.coef(); nm = list(co.index); V = np.asarray(modelo._vcov)
    w = np.zeros(len(nm))
    for t, pi in BUNDLE_W:
        d = _dname(t)
        if d in nm:
            w[nm.index(d)] += pi
    b = float(w @ co.values); se = float(np.sqrt(max(w @ V @ w, 0)))
    return (np.exp(b)-1)*100, (np.exp(b-1.96*se)-1)*100, (np.exp(b+1.96*se)-1)*100

BINARIAS = [c for c in CAPACIDADES if FORMA[c] == 'binaria']
CAPS_EN_MODELO = [c for c in CAPACIDADES if any(c in combo for combo in LISTA_C9)]
print(f"Binarias: {BINARIAS} | capacidades en el modelo: {CAPS_EN_MODELO}")

In [ ]:
# Estabilidad ante cambio de ventana temporal: sin primeros 6 meses, sin últimos 6, y por mitades
pmin, pmax = int(dp['period_id'].min()), int(dp['period_id'].max())
pmed = (pmin + pmax) // 2
ventanas = {
    'sin_6_iniciales': dp['period_id'] > pmin + 6,
    'sin_6_finales':   dp['period_id'] <= pmax - 6,
    'mitad_1':         dp['period_id'] <= pmed,
    'mitad_2':         dp['period_id'] > pmed,
}
res_ventanas = {'base': co_fd}
for nombre, mask in ventanas.items():
    t0 = time.time()
    m = estimar_fd(dp[mask])
    res_ventanas[nombre] = m.coef()
    print(f"{nombre}: {int(mask.sum()):,} filas, {time.time()-t0:.0f}s")
    del m
    gc.collect()

tabla_v = pd.DataFrame(res_ventanas).round(5)
tabla_v['signo_estable'] = np.sign(tabla_v['mitad_1']) == np.sign(tabla_v['mitad_2'])
print("\nCoeficientes por ventana (el signo debe sostenerse en ambas mitades):")
print(tabla_v.to_string())

In [ ]:
# Outcome en unidades físicas: el efecto debe sostenerse; la diferencia es el componente de precio implícito
uc = spark.read.parquet(PATH_MODELO).select('id_cliente', 'period_id', 'unit_cases_total').toPandas()
panel_pd = panel_pd.merge(uc, on=['id_cliente', 'period_id'], how='left')
del uc
gc.collect()
log_uc = np.log(panel_pd['unit_cases_total'].clip(lower=0.01)).astype('float32')
panel_pd = panel_pd.drop(columns=['unit_cases_total'])
panel_pd['log_uc'] = log_uc
del log_uc
base = panel_pd.set_index(['id_cliente', 'period_id'])['log_uc']
idx  = pd.MultiIndex.from_arrays([panel_pd['id_cliente'], panel_pd['period_id'] - 1])
panel_pd['d_uc'] = (panel_pd['log_uc'].values - base.reindex(idx).values).astype('float32')
panel_pd = panel_pd.drop(columns=['log_uc'])
del base, idx
gc.collect()

dp_uc = panel_pd[['id_cliente', 'period_id', 'd_uc'] + D_TERMS].dropna()
m_uc = estimar(f"d_uc ~ {' + '.join(D_TERMS)} | id_cliente + period_id", dp_uc)
co_uc = m_uc.coef()
del m_uc, dp_uc
gc.collect()
print(f"{'término':<42}{'monetario':>12}{'físico':>12}{'precio implícito':>18}")
for d in D_TERMS:
    if d in co_fd.index and d in co_uc.index:
        print(f"{d:<42}{float(co_fd[d]):>12.5f}{float(co_uc[d]):>12.5f}{float(co_fd[d]) - float(co_uc[d]):>18.5f}")

In [ ]:
# Verificación empírica de bad controls: al incluir los mediadores del grafo (frecuencia y volumen),  el efecto de las capacidades debe colapsar; esa caída documenta por qué los mediadores no son controles
med = spark.read.parquet(PATH_MODELO).select('id_cliente', 'period_id', 'ordenes_promedio').toPandas()
panel_pd = panel_pd.merge(med, on=['id_cliente', 'period_id'], how='left')
del med
gc.collect()
freq = panel_pd['ordenes_promedio'].fillna(0).astype('float32')
panel_pd = panel_pd.drop(columns=['ordenes_promedio'])
panel_pd['freq'] = freq
del freq
base = panel_pd.set_index(['id_cliente', 'period_id'])['freq']
idx  = pd.MultiIndex.from_arrays([panel_pd['id_cliente'], panel_pd['period_id'] - 1])
panel_pd['d_freq'] = (panel_pd['freq'].values - base.reindex(idx).values).astype('float32')
panel_pd = panel_pd.drop(columns=['freq'])
del base, idx
gc.collect()

dp_bc = panel_pd[['id_cliente', 'period_id', 'd_y', 'd_freq', 'd_uc'] + D_TERMS].dropna()
m_bc = estimar(f"d_y ~ {' + '.join(D_TERMS + ['d_freq', 'd_uc'])} | id_cliente + period_id", dp_bc)
co_bc = m_bc.coef()
del m_bc, dp_bc
gc.collect()
panel_pd = panel_pd.drop(columns=['d_uc', 'd_freq'])
gc.collect()
print(f"{'término':<42}{'oficial':>12}{'con mediadores':>16}{'caída':>10}")
for d in D_TERMS:
    if d in co_fd.index and d in co_bc.index:
        b0, b1 = float(co_fd[d]), float(co_bc[d])
        caida = (1 - b1 / b0) * 100 if b0 != 0 else float('nan')
        print(f"{d:<42}{b0:>12.5f}{b1:>16.5f}{caida:>+9.1f}%")

In [ ]:
# Placebo de unidades: PDVs never-treated con fecha de adopción ficticia muestreada de la distribución real de adopción de los tratados; el coeficiente placebo debe ser no significativo (p>=0.05)
rng = np.random.RandomState(SEED)
print(f"{'capacidad':<18}{'β placebo':>12}{'SE':>10}{'p-valor':>10}{'lectura':>24}{'PDVs never':>12}")
for cap in BINARIAS:
    g = panel_pd.groupby('id_cliente', observed=True)[cap].max()
    never = g[g == 0].index
    ad = panel_pd.loc[panel_pd[cap] > 0].groupby('id_cliente', observed=True)['period_id'].min()
    del g
    if len(never) == 0 or len(ad) == 0:
        print(f"{cap:<18}{'sin grupo never-treated o sin tratados':>66}")
        continue
    fechas = pd.Series(rng.choice(ad.values, size=len(never)), index=never)
    sub = panel_pd.loc[panel_pd['id_cliente'].isin(set(never)), ['id_cliente', 'period_id', 'd_y']].copy()
    sub['t_falsa'] = sub['id_cliente'].map(fechas).astype('float32')
    sub['placebo'] = (sub['period_id'] >= sub['t_falsa']).astype('float32')
    base = sub.set_index(['id_cliente', 'period_id'])['placebo']
    idx  = pd.MultiIndex.from_arrays([sub['id_cliente'], sub['period_id'] - 1])
    sub['d_placebo'] = (sub['placebo'].values - base.reindex(idx).values).astype('float32')
    sub = sub.dropna(subset=['d_y', 'd_placebo'])
    m = estimar("d_y ~ d_placebo | id_cliente + period_id", sub)
    tdp = m.tidy()
    b  = float(tdp.loc['d_placebo', 'Estimate'])
    se = float(tdp.loc['d_placebo', 'Std. Error'])
    p  = float(tdp.loc['d_placebo', 'Pr(>|t|)'])
    lect = 'OK (no significativo)' if p >= 0.05 else 'ALERTA: significativo'
    print(f"{cap:<18}{b:>12.5f}{se:>10.5f}{p:>10.4f}{lect:>24}{len(never):>12,}")
    del sub, m, tdp, base, idx, ad, fechas
    gc.collect()

In [ ]:
# Precisión estadística: ratio ancho del IC 95 / |coeficiente|; ratio >= 4 marca el término como impreciso
FLAG_PRECISION = []
print(f"{'término':<42}{'coef':>10}{'ancho IC':>12}{'ratio':>8}")
for t in td_fd.index:
    b  = float(td_fd.loc[t, 'Estimate'])
    lo = float(td_fd.loc[t, '2.5%'])
    hi = float(td_fd.loc[t, '97.5%'])
    ratio = (hi - lo) / abs(b) if b != 0 else float('inf')
    marca = '  IMPRECISO' if ratio >= 4 else ''
    if ratio >= 4:
        FLAG_PRECISION.append(t)
    print(f"{t:<42}{b:>10.5f}{hi - lo:>12.5f}{ratio:>8.2f}{marca}")
print(f"\nTérminos imprecisos (ratio >= 4): {len(FLAG_PRECISION)}")

In [ ]:
# Placebo temporal: fecha ficticia 3-6 meses antes de la primera adopción real, estimado solo con el período pre-tratamiento; el coeficiente placebo debe ser no significativo (p>=0.05)
print(f"{'capacidad':<18}{'despl.':>8}{'β placebo':>12}{'IC 95%':>26}{'p-valor':>10}  lectura")
for cap in BINARIAS:
    ad = panel_pd.loc[panel_pd[cap] > 0].groupby('id_cliente', observed=True)['period_id'].min().rename('t_adop')
    if ad.empty:
        continue
    for desp in PLACEBO_DESPLAZAMIENTOS:
        sub = panel_pd[['id_cliente', 'period_id', 'd_y']].merge(ad, left_on='id_cliente', right_index=True, how='inner')
        sub = sub[sub['period_id'] < sub['t_adop']].copy()
        sub['placebo'] = (sub['period_id'] >= sub['t_adop'] + desp).astype('float32')
        base = sub.set_index(['id_cliente', 'period_id'])['placebo']
        idx  = pd.MultiIndex.from_arrays([sub['id_cliente'], sub['period_id'] - 1])
        sub['d_placebo'] = (sub['placebo'].values - base.reindex(idx).values).astype('float32')
        sub = sub.dropna(subset=['d_y', 'd_placebo'])
        if len(sub) < 10_000:
            print(f"{cap:<18}{desp:>+7}m  muestra insuficiente ({len(sub):,} filas)")
            del sub
            continue
        m = estimar("d_y ~ d_placebo | id_cliente + period_id", sub)
        tdp = m.tidy()
        b  = float(tdp.loc['d_placebo', 'Estimate'])
        lo = float(tdp.loc['d_placebo', '2.5%'])
        hi = float(tdp.loc['d_placebo', '97.5%'])
        p  = float(tdp.loc['d_placebo', 'Pr(>|t|)'])
        lect = 'OK (no significativo)' if p >= 0.05 else 'ALERTA: pre-tendencia'
        print(f"{cap:<18}{desp:>+7}m{b:>12.5f}   [{lo:+.5f}; {hi:+.5f}]{p:>10.4f}  {lect}")
        del sub, m, tdp, base, idx
        gc.collect()
    del ad

In [ ]:
# Estudio de eventos por cohortes: coeficientes por período relativo a la adopción (referencia: t-1); pre-adopción cercana a cero valida la identificación; la forma posterior distingue efecto estático de dinámico
K_PRE, K_POST = 6, 6
for cap in BINARIAS:
    ad = panel_pd.loc[panel_pd[cap] > 0].groupby('id_cliente', observed=True)['period_id'].min().rename('t_adop')
    if ad.empty:
        continue
    sub = panel_pd[['id_cliente', 'period_id', 'log_y']].merge(ad, left_on='id_cliente', right_index=True, how='left')
    sub['ev'] = sub['period_id'] - sub['t_adop']
    ks = [k for k in range(-K_PRE, K_POST + 1) if k != -1]
    dummies = []
    for k in ks:
        col = f'ev_{"m" if k < 0 else "p"}{abs(k)}'
        sub[col] = (sub['ev'] == k).astype('float32')
        dummies.append(col)
    m = estimar(f"log_y ~ {' + '.join(dummies)} | id_cliente + period_id", sub)
    tde = m.tidy()
    del m, sub
    gc.collect()
    bs  = [float(tde.loc[c, 'Estimate']) for c in dummies]
    los = [float(tde.loc[c, '2.5%']) for c in dummies]
    his = [float(tde.loc[c, '97.5%']) for c in dummies]
    plt.figure(figsize=(8, 3.5))
    plt.errorbar(ks, bs, yerr=[np.array(bs) - np.array(los), np.array(his) - np.array(bs)],
                 fmt='o-', ms=4, capsize=3)
    plt.axhline(0, color='gray', lw=0.7)
    plt.axvline(-0.5, color='red', lw=0.7, ls='--')
    plt.title(f'Event study {cap} (referencia: t-1)')
    plt.xlabel('meses relativos a la adopción'); plt.ylabel('coef. log')
    plt.tight_layout(); plt.show()
    post_ini = np.mean([b for k, b in zip(ks, bs) if 0 <= k <= 2])
    post_fin = np.mean([b for k, b in zip(ks, bs) if 4 <= k <= 6])
    dinamico = post_fin > post_ini * 1.5 and post_fin - post_ini > 0.01
    print(f"{cap}: post 0-2m {post_ini:+.4f} | post 4-6m {post_fin:+.4f} | "
          f"{'efecto DINÁMICO: requiere diagnóstico de pesos negativos (Goodman-Bacon)' if dinamico else 'efecto estático: sesgo de pesos negativos mínimo'}")
    del ad, tde

In [ ]:
# Placebo por adelantos: el valor t+1 de cada capacidad no debe explicar la venta presente; se lee por la magnitud del efecto implícito, no por el p-valor
D_LEADS = []
for cap in CAPS_EN_MODELO:
    col = termino_forma(cap) if FORMA[cap] != 'cuadratica' else cap
    base_f = panel_pd.set_index(['id_cliente', 'period_id'])[col]
    idx_f  = pd.MultiIndex.from_arrays([panel_pd['id_cliente'], panel_pd['period_id'] + 1])
    lead = pd.Series(base_f.reindex(idx_f).values, dtype='float32')
    panel_pd[f'{cap}_f1'] = lead.values
    base_d = panel_pd.set_index(['id_cliente', 'period_id'])[f'{cap}_f1']
    idx_d  = pd.MultiIndex.from_arrays([panel_pd['id_cliente'], panel_pd['period_id'] - 1])
    panel_pd[f'd_{cap}_f1'] = (panel_pd[f'{cap}_f1'].values - base_d.reindex(idx_d).values).astype('float32')
    D_LEADS.append(f'd_{cap}_f1')
    del base_f, idx_f, base_d, idx_d, lead
gc.collect()

dp_lead = panel_pd[['id_cliente', 'period_id', 'd_y'] + D_TERMS + D_LEADS].dropna()
m_lead = estimar(f"d_y ~ {' + '.join(D_TERMS + D_LEADS)} | id_cliente + period_id", dp_lead)
co_l = m_lead.coef()
del m_lead, dp_lead
panel_pd = panel_pd.drop(columns=[f'{cap}_f1' for cap in CAPS_EN_MODELO] + D_LEADS)
gc.collect()

print(f"{'capacidad':<18}{'β adelanto':>12}{'efecto implícito':>18}")
for cap in CAPS_EN_MODELO:
    d = f'd_{cap}_f1'
    if d not in co_l.index:
        continue
    b = float(co_l[d])
    ref_l = REF[cap] if FORMA[cap] != 'cuadratica' else float(panel_pd.loc[panel_pd[cap] > 0, cap].median())
    ef = (np.exp(b * ref_l) - 1) * 100
    print(f"{cap:<18}{b:>12.5f}{ef:>+16.2f}%")

In [ ]:
# Sensibilidad al tipo de error estándar: HC1 vs cluster por PDV; un término significativo solo bajo uno de los esquemas no debe reportarse como significativo
if RUN_R_CLUSTER:
    t0 = time.time()
    m_cl = estimar_fd(dp, vcov_tipo='CRV1')
    td_c = m_cl.tidy()
    del m_cl
    gc.collect()
    print(f"CRV1 estimado en {time.time()-t0:.0f}s")
    if VCOV_TIPO == 'HC1':
        td_h = td_fd
    else:
        m_h = estimar_fd(dp, vcov_tipo='HC1')
        td_h = m_h.tidy()
        del m_h
        gc.collect()
    print(f"{'término':<42}{'β':>10}{'p HC1':>10}{'p CRV1':>10}  lectura")
    for t in td_h.index:
        if t not in td_c.index:
            continue
        ph, pc = float(td_h.loc[t, 'Pr(>|t|)']), float(td_c.loc[t, 'Pr(>|t|)'])
        lect = 'robusto' if (ph < 0.05) == (pc < 0.05) else 'SOLO bajo un esquema: no reportar como significativo'
        print(f"{t:<42}{float(td_h.loc[t, 'Estimate']):>10.5f}{ph:>10.4f}{pc:>10.4f}  {lect}")
else:
    print("Omitido (RUN_R_CLUSTER=False); activar para el reporte final")

# Heterogeneidad del bundle por segmento

In [ ]:
# Bundle de largo plazo re-estimado por segmento de tamaño de cliente y por subcanal
for col_seg in ['tamano_cliente', 'subcanal']:
    print(f"\nHeterogeneidad por {col_seg}")
    for seg in dp[col_seg].dropna().unique():
        d = dp[dp[col_seg] == seg]
        n_pdvs = d['id_cliente'].nunique()
        if n_pdvs < 20_000:
            del d
            continue
        m = estimar_fd(d)
        e, lo, hi = bundle_de(m)
        print(f"  {str(seg):<24} bundle largo {e:+.1f}%  [{lo:+.1f}; {hi:+.1f}]   PDVs {n_pdvs:,}")
        del m, d
        gc.collect()